# 01. Supervised Finetuning Chat Masks Neftune and Sequence Packing

- **Category:** 08_learning_paradigms_rl_and_alignment / 03_llm_alignment_and_reasoning
- **Lineage & Citations:** Ouyang et al. (2022 InstructGPT); Jain et al. (2023 NEFTune); Ding et al. (2024 Sequence Packing / FlashAttention VarLen); OpenAI / Meta (ChatML and Structured Delimiters).
- **Purpose:** First-principles mathematical implementation and empirical verification of post-training SFT primitives: chat template completion-only loss masking, NEFTune embedding noise injection, 2D block-diagonal causal masking for sequence packing, and position ID resetting.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [08. Successor Representations Forward Backward Zero Shot RL and DIAYN](../02_reinforcement_learning_and_world_models/08_successor_representations_forward_backward_zero_shot_rl_and_diayn.ipynb) $\to$ `🔗 Downstream Syntheses:` [02. RLHF Reward Modeling and PPO vs RLOO](02_rlhf_reward_modeling_and_ppo_vs_rloo.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to Use**  
> Never train conversational SFT without completion loss masking; evaluating cross-entropy on prompt/system tokens incentivizes the model to generate prompts rather than answers. In packed sequences, never use a standard triangular mask without document segment checking, as cross-document attention contamination will corrupt generation.

## Mathematical Foundations & Governing Equations

### 1. NEFTune Noise Injection
$$
\mathbf{e}_{\text{noisy}} = \mathbf{e} + \frac{\alpha}{\sqrt{L \cdot D}} \boldsymbol{\epsilon}, \quad \boldsymbol{\epsilon} \sim \text{Uniform}(-1, 1)^{B \times L \times D}
$$

### 2. Completion-Only Masked Cross-Entropy Loss
$$
\mathcal{L}_{\text{SFT}} = -\frac{\sum_{b=1}^B \sum_{t=1}^L m_{b, t} \ln P_\theta(y_{b, t} \mid y_{b, <t})}{\sum_{b=1}^B \sum_{t=1}^L m_{b, t} + \epsilon}
$$

### 3. Packed Block-Diagonal Causal Attention Mask
$$
M_{i, j} = \begin{cases} 1.0 & \text{if } j \le i \text{ and } \text{seg}[i] = \text{seg}[j] \\ 0.0 & \text{otherwise} \end{cases}
$$

### 4. Packed Position ID Resetting
$$
\text{pos}[i] = i - \text{start}(\text{seg}[i])
$$

### Asymptotic Time & Memory Complexity
- NEFTune Noise Injection: $\mathcal{O}(B \cdot L \cdot D)$ time and memory.
- Masked Completion Cross-Entropy: $\mathcal{O}(B \cdot L \cdot V)$ time and memory.
- Block-Diagonal Causal Mask: $\mathcal{O}(L^2)$ time and memory.
- Position ID Calculation: $\mathcal{O}(L)$ time and memory.

In [ ]:
# Setup and seed verification environment
import math
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)

# Locate the golden checkpoint
CKPT_REL = Path("weights/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/01_supervised_finetuning_chat_masks_neftune_and_sequence_packing.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, w: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return x @ w + b

def neftune_add_noise(
    embeddings: torch.Tensor,
    noise: torch.Tensor,
    alpha: float = 5.0,
) -> torch.Tensor:
    b, seq_len, d_dim = embeddings.shape
    scale = alpha / math.sqrt(seq_len * d_dim)
    return embeddings + scale * noise

def masked_completion_cross_entropy(
    logits: torch.Tensor,
    targets: torch.Tensor,
    loss_mask: torch.Tensor,
    eps: float = 1e-9,
) -> torch.Tensor:
    log_probs = F.log_softmax(logits, dim=-1)
    target_log_probs = log_probs.gather(dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)
    nll = -target_log_probs
    masked_nll = nll * loss_mask
    return torch.sum(masked_nll) / (torch.sum(loss_mask) + eps)

def packed_block_diagonal_causal_mask(
    segment_ids: torch.Tensor,
) -> torch.Tensor:
    seq_len = segment_ids.shape[0]
    indices = torch.arange(seq_len, device=segment_ids.device)
    causal_mask = indices.unsqueeze(0) <= indices.unsqueeze(1)
    same_segment = segment_ids.unsqueeze(0) == segment_ids.unsqueeze(1)
    return (causal_mask & same_segment).float()

def packed_position_ids(
    segment_ids: torch.Tensor,
) -> torch.Tensor:
    seq_len = segment_ids.shape[0]
    pos_ids = torch.zeros(seq_len, dtype=torch.long, device=segment_ids.device)
    current_pos = 0
    for i in range(seq_len):
        if i > 0 and segment_ids[i] != segment_ids[i - 1]:
            current_pos = 0
        pos_ids[i] = current_pos
        current_pos += 1
    return pos_ids

In [ ]:
# Multi-track numerical parity assertions
y = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
err_anchor = float(torch.max(torch.abs(y - weights["test_output"])))
assert err_anchor < 1e-5, f"Linear anchor parity error: {err_anchor:.2e}"

noisy = neftune_add_noise(weights["embeddings"], weights["noise"], alpha=5.0)
err_nef = float(torch.max(torch.abs(noisy - weights["neftune_noisy_ref"])))
assert err_nef < 1e-5, f"NEFTune noise parity error: {err_nef:.2e}"

loss = masked_completion_cross_entropy(weights["logits"], weights["targets"], weights["loss_mask"])
err_loss = float(torch.abs(loss - weights["masked_loss_ref"]))
assert err_loss < 1e-5, f"Masked completion loss parity error: {err_loss:.2e}"

mask = packed_block_diagonal_causal_mask(weights["segment_ids"])
err_mask = float(torch.max(torch.abs(mask - weights["packed_mask_ref"])))
assert err_mask < 1e-5, f"Packed block-diagonal mask parity error: {err_mask:.2e}"

pos = packed_position_ids(weights["segment_ids"])
err_pos = float(torch.max(torch.abs(pos.float() - weights["packed_positions_ref"].float())))
assert err_pos < 1e-5, f"Packed position IDs parity error: {err_pos:.2e}"

print(f"All parity assertions passed successfully (max error {max(err_anchor, err_nef, err_loss, err_mask, err_pos):.2e})")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_nef = benchmark_fn(
    lambda: neftune_add_noise(weights["embeddings"], weights["noise"]),
    iters=100,
    name="NEFTune Noise Injection",
)
b_loss = benchmark_fn(
    lambda: masked_completion_cross_entropy(weights["logits"], weights["targets"], weights["loss_mask"]),
    iters=100,
    name="Masked Completion Cross-Entropy",
)
b_mask = benchmark_fn(
    lambda: packed_block_diagonal_causal_mask(weights["segment_ids"]),
    iters=100,
    name="Block-Diagonal Attention Mask",
)
b_pos = benchmark_fn(
    lambda: packed_position_ids(weights["segment_ids"]),
    iters=100,
    name="Packed Position IDs",
)

print(format_benchmark_table([b_nef, b_loss, b_mask, b_pos]))

In [ ]:
# Failure Mode Contrast: Unmasked Training vs Completion-Only Loss Masking
# Suppose a sequence has a prompt token at index 0 and completion token at index 1
logits_toy = torch.tensor([[2.0, -1.0], [0.0, 3.0]], requires_grad=True)
targets_toy = torch.tensor([0, 1])

# Case 1: Unmasked training (loss evaluated across prompt token at index 0 and completion at index 1)
loss_unmasked = F.cross_entropy(logits_toy, targets_toy)
grad_unmasked = torch.autograd.grad(loss_unmasked, logits_toy, retain_graph=True)[0]
grad_prompt_unmasked = torch.norm(grad_unmasked[0]).item()

# Case 2: Completion-only masked training (mask = [0.0, 1.0])
mask_toy = torch.tensor([0.0, 1.0])
loss_masked = masked_completion_cross_entropy(logits_toy.unsqueeze(0), targets_toy.unsqueeze(0), mask_toy.unsqueeze(0))
grad_masked = torch.autograd.grad(loss_masked, logits_toy)[0]
grad_prompt_masked = torch.norm(grad_masked[0]).item()

print(f"Unmasked Prompt Token Gradient Norm:        {grad_prompt_unmasked:.4f} (Forced to memorize prompt tokens!)")
print(f"Completion-Masked Prompt Gradient Norm:    {grad_prompt_masked:.4f} (Zero prompt gradient, pure response learning!)")

assert grad_prompt_unmasked > 0.01, f"Expected non-zero unmasked prompt gradient (got {grad_prompt_unmasked})"
assert abs(grad_prompt_masked - 0.0) < 1e-5, f"Expected exactly zero masked prompt gradient (got {grad_prompt_masked})"
print("Pathological failure mode contrast successfully verified")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Completion Masking**: Preserves prompt invariance by driving loss gradients exclusively from assistant tokens, eliminating prompt memorization.
2. **NEFTune Regularization**: Adding scaled embedding noise breaks superficial template memorization without requiring additional synthetic training data.
3. **Zero-Pad Sequence Packing**: Multipack grouping eliminates computational waste on `<pad>` tokens while block-diagonal masking preserves strict causal boundaries.
4. **Rotary Position Resetting**: Sub-sequence position resets prevent RoPE relative position corruption across packed sequences.

---
### Cross-Domain Generalisation Matrix
- **Conversational Assistants**: Completion masking ensures robust conversational turn-taking without auto-reply hallucination.
- **Code Generation SFT**: Sequence packing eliminates padding over thousands of varying-length function bodies, tripling training throughput.
- **Medical Reasoning & Document QA**: Segmented block-diagonal attention enables packing heterogeneous electronic health records into unified 32k context batches without information leakage.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [01_supervised_finetuning_chat_masks_neftune_and_sequence_packing.md](../../../markdown/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/01_supervised_finetuning_chat_masks_neftune_and_sequence_packing.md)
- **PyTorch 2.x Module:** [01_supervised_finetuning_chat_masks_neftune_and_sequence_packing.py](../../../pytorch2/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/01_supervised_finetuning_chat_masks_neftune_and_sequence_packing.py)
- **Functional JAX Module:** [01_supervised_finetuning_chat_masks_neftune_and_sequence_packing.py](../../../jax/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/01_supervised_finetuning_chat_masks_neftune_and_sequence_packing.py)
- **Native Rust Candle Track:** [01_supervised_finetuning_chat_masks_neftune_and_sequence_packing.rs](../../../candle/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/01_supervised_finetuning_chat_masks_neftune_and_sequence_packing.rs)
- **Hardware Kernel (CPU / WGSL):** [01_supervised_finetuning_chat_masks_neftune_and_sequence_packing.rs](../../../kernels/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/01_supervised_finetuning_chat_masks_neftune_and_sequence_packing.rs)
- **Golden SafeTensors:** [01_supervised_finetuning_chat_masks_neftune_and_sequence_packing.safetensors](../../../weights/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/01_supervised_finetuning_chat_masks_neftune_and_sequence_packing.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 08. Successor Representations Forward Backward Zero Shot RL and DIAYN](../02_reinforcement_learning_and_world_models/08_successor_representations_forward_backward_zero_shot_rl_and_diayn.ipynb) | [Master Index](../../../README.md#08_learning_paradigms_rl_and_alignment) | [Next: 02. RLHF Reward Modeling and PPO vs RLOO](02_rlhf_reward_modeling_and_ppo_vs_rloo.ipynb) &rarr;